# Capped hierarchical CV experiments

Runs the expanded reconciliation recipe on a capped M5 sample and scores bottom-level WRMSSE. Keep `n_windows=1` for local iteration; raise it only after the method list is stable.

In [1]:
from pathlib import Path

import pandas as pd

from m5.config import SETTINGS, set_global_seed
from m5.cv import cv_from_recipe
from m5.evaluation import compute_components, wrmsse_for_models
from m5.recipes import Recipe

set_global_seed()

long_path = SETTINGS.processed_dir / "long.parquet"
recipe_path = Path("configs/m5/hier_experiments.yaml")
artifact_path = SETTINGS.artifacts_dir / "hierarchical" / "cv_hier_experiments.parquet"
recipe = Recipe.from_yaml(recipe_path)
recipe

Recipe(task=TaskRecipe(name='m5', id_col='unique_id', time_col='ds', target_col='y', freq='D', horizon=28, static_cols=['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'], dynamic_cols=[], drop_leading_zeros=True), model=HierRecipe(kind='hier', base_model=StatsModelSpec(name='Theta', alias='Theta', season_length=7), reconcilers=['BU', 'MinT_OLS', 'MinT_WLS_struct', 'MinT_WLS_var', 'MinT_shrink', 'ERM_closed', 'ERM_reg_bu']), cv=CVRecipe(n_windows=1, step_size=None))

Build capped data before running:

```bash
M5_N_SERIES=500 M5_LAST_N_DAYS=200 make prep
```

For a faster smoke run in the notebook, override `h` and `n_windows` below.

In [2]:
df = pd.read_parquet(long_path)
df["ds"] = pd.to_datetime(df["ds"])
df.shape, df["unique_id"].nunique(), df["ds"].min(), df["ds"].max()

((1004671, 18),
 5000,
 Timestamp('2015-11-04 00:00:00'),
 Timestamp('2016-05-22 00:00:00'))

In [3]:
cv_df = cv_from_recipe(
    recipe,
    df,
    h=28,
    n_windows=1,
)
cv_df.head()

07:38:00 | INFO    | m5.cv:hier_cv:102 - hier_cv: h=28 n_windows=1 step=28 levels=12 series=11803
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[

,unique_id,ds,Theta,Theta/BottomUp,Theta/MinTrace_method-ols,Theta/MinTrace_method-mint_shrink,cutoff,y
0,FOODS_1_001_CA_1,2016-04-25,1.001816,1.001816,0.833393,0.979775,2016-04-24,2.0
1,FOODS_1_001_CA_1,2016-04-26,1.002816,1.002816,0.800524,0.972670,2016-04-24,0.0
2,FOODS_1_001_CA_1,2016-04-27,1.003816,1.003816,0.758972,0.972059,2016-04-24,0.0
3,FOODS_1_001_CA_1,2016-04-28,1.004816,1.004816,0.861146,0.980167,2016-04-24,0.0
4,FOODS_1_001_CA_1,2016-04-29,1.005816,1.005816,0.941286,0.980747,2016-04-24,0.0


In [4]:
artifact_path.parent.mkdir(parents=True, exist_ok=True)
cv_df.to_parquet(artifact_path, index=False)
artifact_path

PosixPath('/home/ricka/Git/GitHub/M5/artifacts/hierarchical/cv_hier_experiments.parquet')

In [5]:
train_pre_cv = df[df["ds"] < cv_df["ds"].min()]
components = compute_components(train_pre_cv)
truth = cv_df[["unique_id", "ds", "y"]]
scores = wrmsse_for_models(truth, cv_df, components).rename_axis("model").reset_index(name="wrmsse")
scores.sort_values("wrmsse")

,model,wrmsse
0,Theta/MinTrace_method-ols,0.801326
1,Theta/MinTrace_method-mint_shrink,0.803263
2,Theta/BottomUp,0.803586
3,Theta,0.803586


In [6]:
forecast_cols = [c for c in cv_df.columns if c not in {"unique_id", "ds", "cutoff", "y"}]
error_by_h = cv_df.copy()
error_by_h["h"] = error_by_h.groupby(["unique_id", "cutoff"]).cumcount() + 1
rows = []
for col in forecast_cols:
    err = (error_by_h[col] - error_by_h["y"]).abs()
    rows.append(error_by_h.assign(abs_error=err).groupby("h", as_index=False)["abs_error"].mean().assign(model=col))
pd.concat(rows, ignore_index=True).pivot(index="h", columns="model", values="abs_error")

model,Theta,Theta/BottomUp,Theta/MinTrace_method-mint_shrink,Theta/MinTrace_method-ols
h,,,,
1,0.947648,0.947648,0.938869,0.936840
2,0.903934,0.903934,0.893669,0.888017
3,0.965577,0.965577,0.950556,0.950288
4,0.944958,0.944958,0.918918,0.923213
5,0.988481,0.988481,0.985600,0.976672
6,1.045630,1.045629,1.067889,1.073763
7,1.127470,1.127470,1.135714,1.147216
8,1.023797,1.023797,1.020464,1.014405
9,1.034816,1.034816,1.038773,1.025472


## Reading the result

- If BottomUp wins, the bottom-level Theta forecasts carry most of the useful signal and reconciliation should mainly enforce aggregate consistency.
- If WLS variance or MinTrace shrinkage wins, residual scale differs materially across levels, stores, or product groups.
- If ERM wins on one capped sample but not others, treat it as overfit until it repeats over more windows and larger samples.

Promote a method only after rerunning with at least `M5_N_WINDOWS=3` and checking per-level/segment reports with `uv run m5 score`.